# 06 — Week 2 Projections

This notebook converts the updated weekly team-strength ratings into game level projections for the selected target week.

The original preseason notebooks and preseason projections remain frozen.

For each target week matchup, this notebook:

- loads the updated weekly team strengths from Notebook 04
- preserves the original preseason Week 2 projection as a baseline
- recalculates expected margin using weekly strength, historical home field advantage, and rest
- converts expected margin into win probability using historical NFL margin variability
- carries forward the frozen preseason scoring baseline created for Week 1
- adds the new in season offensive and defensive scoring evidence
- produces projected scores, model spreads, ATS comparisons, totals comparisons, and final output tables
- saves all outputs only inside the weekly layer

The market is used only for comparison. It never enters the model projection itself.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
from sklearn.linear_model import LinearRegression


## Paths and Weekly Settings

For Week 2, `TARGET_WEEK = 2`.

Future weeks use the same notebook by changing the target week after the preceding weekly notebooks have been rerun.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = PROJECT_ROOT / "weekly_projections" / "outputs"

SEASON = 2026
TARGET_WEEK = 4

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}"
)

WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

INJURY_ADJUSTED_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

TARGET_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

PRESEASON_BASELINE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

WEEK1_PROJECTIONS_PATH = (
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)

# Load Weekly Inputs

Notebook 04 supplies the updated ratings. Notebook 02 supplies the target week schedule and frozen preseason comparison.


In [3]:
weekly_strength = pd.read_parquet(WEEKLY_STRENGTH_PATH)

injury_adjusted_strength = pd.read_parquet(
    INJURY_ADJUSTED_STRENGTH_PATH
)

target_schedule = pd.read_parquet(
    TARGET_SCHEDULE_PATH
)

preseason_baseline = pd.read_parquet(
    PRESEASON_BASELINE_PATH
)

target_schedule = target_schedule[
    target_schedule["home_score"].isna()
    & target_schedule["away_score"].isna()
].copy()


preseason_baseline = preseason_baseline[
    preseason_baseline["game_id"].isin(
        target_schedule["game_id"]
    )
].copy()


print(
    "Sequential weekly strengths:",
    weekly_strength.shape
)

print(
    "Injury-adjusted strengths:",
    injury_adjusted_strength.shape
)

print(
    "Remaining target-week games:",
    target_schedule.shape
)

print(
    "Frozen preseason games:",
    preseason_baseline.shape
)

assert len(weekly_strength) == 32
assert weekly_strength["team"].nunique() == 32

assert len(injury_adjusted_strength) == 32
assert injury_adjusted_strength["team"].nunique() == 32

assert len(target_schedule) == len(
    preseason_baseline
)

required_injury_columns = [
    "team",
    "weekly_team_strength",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

missing_injury_columns = [
    col
    for col in required_injury_columns
    if col not in injury_adjusted_strength.columns
]

assert not missing_injury_columns, (
    "Missing injury-adjusted columns: "
    + ", ".join(missing_injury_columns)
)


print(
    "Using sequential ratings as the pre-injury baseline "
    "and injury-adjusted ratings for Week "
    f"{TARGET_WEEK} projections."
)

Sequential weekly strengths: (32, 25)
Injury-adjusted strengths: (32, 30)
Remaining target-week games: (16, 46)
Frozen preseason games: (16, 16)
Using sequential ratings as the pre-injury baseline and injury-adjusted ratings for Week 4 projections.


### Injury adjusted game strength

For Week 2, the `home_weekly_team_strength` and `away_weekly_team_strength` values now include the conservative injury adjustment from Notebook 05.

The injury adjustment remains visible separately in the saved team strength file, so we can distinguish:

**preseason → performance update → injury update → final Week 2 rating**


# Attach Updated Team Strength

The matchup calculation is identical in structure to the original preseason game prediction model. The only change is the source of team strength:

`preseason team_strength` → `weekly_team_strength`

This isolates the in season update without changing the original model.


In [4]:
strength_lookup = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength",
        "preseason_team_strength",
        "team_strength_change",
        "cumulative_offense_adjustment",
        "cumulative_defense_adjustment"
    ]
].copy()


games = target_schedule.copy()

home_lookup = strength_lookup.rename(
    columns={
        "team": "home_team",

        "weekly_team_strength":
            "home_pre_injury_team_strength",

        "injury_adjustment":
            "home_injury_adjustment",

        "injury_adjusted_team_strength":
            "home_weekly_team_strength",

        "preseason_team_strength":
            "home_preseason_team_strength",

        "team_strength_change":
            "home_strength_change",

        "cumulative_offense_adjustment":
            "home_offense_adjustment",

        "cumulative_defense_adjustment":
            "home_defense_adjustment"
    }
)

away_lookup = strength_lookup.rename(
    columns={
        "team": "away_team",

        "weekly_team_strength":
            "away_pre_injury_team_strength",

        "injury_adjustment":
            "away_injury_adjustment",

        "injury_adjusted_team_strength":
            "away_weekly_team_strength",

        "preseason_team_strength":
            "away_preseason_team_strength",

        "team_strength_change":
            "away_strength_change",

        "cumulative_offense_adjustment":
            "away_offense_adjustment",

        "cumulative_defense_adjustment":
            "away_defense_adjustment"
    }
)


games = (
    games
    .merge(
        home_lookup,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_lookup,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

games["weekly_neutral_strength_diff"] = (
    games["home_weekly_team_strength"]
    - games["away_weekly_team_strength"]
)

games["pre_injury_neutral_strength_diff"] = (
    games["home_pre_injury_team_strength"]
    - games["away_pre_injury_team_strength"]
)


games["preseason_neutral_strength_diff"] = (
    games["home_preseason_team_strength"]
    - games["away_preseason_team_strength"]
)

assert games[
    "home_weekly_team_strength"
].notna().all()

assert games[
    "away_weekly_team_strength"
].notna().all()

assert games[
    "home_injury_adjustment"
].notna().all()

assert games[
    "away_injury_adjustment"
].notna().all()

# Home Field and Rest

These effects are recalculated using the same historical 2015–2025 methodology as the frozen preseason game prediction notebook.

They are not retuned using Week 1.


In [5]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()
rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print("Historical HFA:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest points/day:", round(REST_POINT_VALUE, 3))


Historical HFA: 1.764
Rest points/day: 0.181


In [6]:
games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"]
    - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"]
    * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["weekly_neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)


# Win Probabilities

Historical 2015–2025 game-margin variability is retained exactly as the uncertainty scale. Week 1 is not used to recalibrate probability confidence.


In [7]:
historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.50,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

print("Historical margin SD:", round(historical_margin_std, 3))


Historical margin SD: 14.201


# Preseason vs Weekly Margin Movement

The frozen preseason Week 2 projection is attached for diagnostics only.

This makes it possible to see exactly how much the Week 1 evidence changed each Week 2 matchup.


In [8]:
preseason_compare = preseason_baseline[
    [
        "game_id",
        "expected_home_margin",
        "predicted_winner",
        "predicted_win_probability"
    ]
].rename(columns={
    "expected_home_margin": "preseason_expected_home_margin",
    "predicted_winner": "preseason_predicted_winner",
    "predicted_win_probability": "preseason_predicted_win_probability"
})

games = games.merge(
    preseason_compare,
    on="game_id",
    how="left",
    validate="one_to_one"
)

games["margin_change_from_preseason"] = (
    games["expected_home_margin"]
    - games["preseason_expected_home_margin"]
)

display(
    games[
        [
            "away_team",
            "home_team",
            "preseason_expected_home_margin",
            "expected_home_margin",
            "margin_change_from_preseason",
            "preseason_predicted_winner",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ].round(3)
)


,away_team,home_team,preseason_expected_home_margin,expected_home_margin,margin_change_from_preseason,preseason_predicted_winner,predicted_winner,predicted_win_probability
0,PIT,CLE,-2.931,-2.551,0.380,PIT,PIT,0.571
1,ARI,NYG,1.986,-0.948,-2.933,NYG,ARI,0.527
2,DAL,HOU,4.960,2.333,-2.627,HOU,HOU,0.565
3,DEN,SF,0.561,1.587,1.026,SF,SF,0.544
4,DET,CAR,-6.788,-4.978,1.810,DET,DET,0.637
5,GB,TB,-0.715,-1.471,-0.756,GB,GB,0.541
6,IND,WAS,-2.534,-4.425,-1.891,IND,IND,0.622
7,JAX,CIN,-0.689,-1.503,-0.814,JAX,JAX,0.542
8,KC,LV,-5.232,-3.825,1.407,KC,KC,0.606
9,LAC,SEA,5.216,9.012,3.796,SEA,SEA,0.737


# Frozen Preseason Scoring Ratings

Week 1's saved projection artifact contains the preseason offensive/defensive scoring ratings and personnel scoring adjustments for all 32 teams.

Those values were created before the season and are reused here as the frozen scoring prior. We do not rebuild or modify the preseason notebooks.

The new weekly offense/defense signals are added separately afterward.


In [9]:
week1_frozen = pd.read_parquet(
    WEEK1_PROJECTIONS_PATH
)

home_scoring = week1_frozen[
    [
        "home_team",
        "home_projected_off_ppg",
        "home_projected_def_ppg",
        "home_off_personnel_z",
        "home_def_personnel_z"
    ]
].rename(columns={
    "home_team": "team",
    "home_projected_off_ppg": "preseason_projected_off_ppg",
    "home_projected_def_ppg": "preseason_projected_def_ppg",
    "home_off_personnel_z": "off_personnel_z",
    "home_def_personnel_z": "def_personnel_z"
})

away_scoring = week1_frozen[
    [
        "away_team",
        "away_projected_off_ppg",
        "away_projected_def_ppg",
        "away_off_personnel_z",
        "away_def_personnel_z"
    ]
].rename(columns={
    "away_team": "team",
    "away_projected_off_ppg": "preseason_projected_off_ppg",
    "away_projected_def_ppg": "preseason_projected_def_ppg",
    "away_off_personnel_z": "off_personnel_z",
    "away_def_personnel_z": "def_personnel_z"
})

scoring_prior = (
    pd.concat([home_scoring, away_scoring], ignore_index=True)
    .groupby("team", as_index=False)
    .agg(
        preseason_projected_off_ppg=("preseason_projected_off_ppg", "mean"),
        preseason_projected_def_ppg=("preseason_projected_def_ppg", "mean"),
        off_personnel_z=("off_personnel_z", "mean"),
        def_personnel_z=("def_personnel_z", "mean")
    )
)

assert len(scoring_prior) == 32
display(scoring_prior.head())


,team,preseason_projected_off_ppg,preseason_projected_def_ppg,off_personnel_z,def_personnel_z
0,ARI,22.191473,24.425844,-0.763765,-1.345326
1,ATL,22.032425,23.261103,0.507936,0.513715
2,BAL,25.007820,22.580763,0.442967,0.969729
3,BUF,25.942602,22.289034,1.060099,0.231615
4,CAR,20.533474,23.731353,-1.506345,-0.346889


In [10]:
completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

completed_games = pd.read_parquet(
    completed_games_path
).copy()

completed_games = completed_games[
    completed_games["week"] < TARGET_WEEK
].copy()

home_scoring_actual = completed_games[
    [
        "home_team",
        "home_score",
        "away_score"
    ]
].rename(
    columns={
        "home_team": "team",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_scoring_actual = completed_games[
    [
        "away_team",
        "away_score",
        "home_score"
    ]
].rename(
    columns={
        "away_team": "team",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_scoring_actual = pd.concat(
    [
        home_scoring_actual,
        away_scoring_actual
    ],
    ignore_index=True
)

inseason_scoring = (
    team_scoring_actual
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean")
    )
)

assert len(inseason_scoring) == 32
assert (
    inseason_scoring["games_played"] == 3
).all()

display(
    inseason_scoring
    .sort_values("actual_off_ppg", ascending=False)
    .round(2)
)

,team,games_played,actual_off_ppg,actual_def_ppg
3,BUF,3,33.67,26.00
28,SF,3,32.67,16.67
10,DET,3,31.00,31.67
2,BAL,3,30.67,26.00
4,CAR,3,29.67,27.67
5,CHI,3,29.67,17.67
15,KC,3,29.33,16.67
8,DAL,3,29.33,27.33
18,LV,3,29.33,18.00
14,JAX,3,27.33,12.00


In [11]:
scoring_expectations = scoring_prior[
    [
        "team",
        "preseason_projected_off_ppg",
        "preseason_projected_def_ppg"
    ]
].copy()

home_games = completed_games[
    [
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_games = home_games.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_games = completed_games[
    [
        "week",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_games = away_games.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

opponent_defense = scoring_expectations[
    [
        "team",
        "preseason_projected_def_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_def_ppg":
            "opponent_expected_def_ppg"
    }
)

opponent_offense = scoring_expectations[
    [
        "team",
        "preseason_projected_off_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_off_ppg":
            "opponent_expected_off_ppg"
    }
)

team_games = (
    team_games
    .merge(
        opponent_defense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
    .merge(
        opponent_offense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
)

league_preseason_off_ppg = (
    scoring_prior[
        "preseason_projected_off_ppg"
    ].mean()
)

league_preseason_def_ppg = (
    scoring_prior[
        "preseason_projected_def_ppg"
    ].mean()
)

team_games["adjusted_points_for"] = (
    team_games["points_for"]
    + (
        league_preseason_def_ppg
        - team_games["opponent_expected_def_ppg"]
    )
)

team_games["adjusted_points_against"] = (
    team_games["points_against"]
    + (
        league_preseason_off_ppg
        - team_games["opponent_expected_off_ppg"]
    )
)

inseason_scoring_adjusted = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean"),
        adjusted_off_ppg=(
            "adjusted_points_for",
            "mean"
        ),
        adjusted_def_ppg=(
            "adjusted_points_against",
            "mean"
        )
    )
)

assert len(inseason_scoring_adjusted) == 32

display(
    inseason_scoring_adjusted[
        [
            "team",
            "actual_off_ppg",
            "adjusted_off_ppg",
            "actual_def_ppg",
            "adjusted_def_ppg"
        ]
    ]
    .sort_values(
        "adjusted_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,actual_off_ppg,adjusted_off_ppg,actual_def_ppg,adjusted_def_ppg
3,BUF,33.67,34.45,26.00,24.99
28,SF,32.67,32.25,16.67,16.35
10,DET,31.00,30.79,31.67,32.09
5,CHI,29.67,30.15,17.67,18.48
2,BAL,30.67,29.92,26.00,25.55
18,LV,29.33,29.63,18.00,18.92
4,CAR,29.67,29.46,27.67,28.85
15,KC,29.33,29.45,16.67,16.27
8,DAL,29.33,28.73,27.33,27.19
14,JAX,27.33,28.02,12.00,12.70


In [12]:
scoring_model = scoring_prior.merge(
    inseason_scoring_adjusted,
    on="team",
    how="left",
    validate="one_to_one"
)

PRESEASON_EQUIVALENT_GAMES = 5.0

scoring_model["preseason_weight"] = (
    PRESEASON_EQUIVALENT_GAMES
    / (
        PRESEASON_EQUIVALENT_GAMES
        + scoring_model["games_played"]
    )
)

scoring_model["inseason_weight"] = (
    1.0
    - scoring_model["preseason_weight"]
)

scoring_model["current_off_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_off_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_off_ppg"]
)

scoring_model["current_def_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_def_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_def_ppg"]
)

assert len(scoring_model) == 32

assert scoring_model[
    [
        "current_off_ppg",
        "current_def_ppg"
    ]
].notna().all().all()

display(
    scoring_model[
        [
            "team",
            "preseason_projected_off_ppg",
            "adjusted_off_ppg",
            "current_off_ppg",
            "preseason_projected_def_ppg",
            "adjusted_def_ppg",
            "current_def_ppg",
            "preseason_weight",
            "inseason_weight"
        ]
    ]
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,preseason_projected_off_ppg,adjusted_off_ppg,current_off_ppg,preseason_projected_def_ppg,adjusted_def_ppg,current_def_ppg,preseason_weight,inseason_weight
3,BUF,25.94,34.45,29.13,22.29,24.99,23.30,0.62,0.38
10,DET,26.28,30.79,27.97,23.05,32.09,26.44,0.62,0.38
28,SF,24.31,32.25,27.29,22.70,16.35,20.32,0.62,0.38
2,BAL,25.01,29.92,26.85,22.58,25.55,23.69,0.62,0.38
8,DAL,24.71,28.73,26.22,24.75,27.19,25.67,0.62,0.38
5,CHI,23.20,30.15,25.81,23.18,18.48,21.42,0.62,0.38
14,JAX,23.95,28.02,25.47,22.47,12.70,18.80,0.62,0.38
6,CIN,23.96,27.25,25.19,24.55,20.86,23.16,0.62,0.38
15,KC,22.39,29.45,25.04,21.52,16.27,19.55,0.62,0.38
13,IND,24.31,25.06,24.60,23.56,29.79,25.90,0.62,0.38


# Weekly Scoring Update

The in-season offensive and defensive adjustments from Notebook 04 are already shrunk for sample size.

They are added to the frozen scoring prior:

- positive offensive adjustment raises expected scoring
- positive defensive adjustment represents points saved, so it lowers projected points allowed

This keeps the Week 1 scoring evidence separate and interpretable.


In [13]:
scoring_update = scoring_model[
    [
        "team",
        "current_off_ppg",
        "current_def_ppg"
    ]
].copy()

assert len(scoring_update) == 32
assert scoring_update["team"].nunique() == 32

display(
    scoring_update
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,current_off_ppg,current_def_ppg
3,BUF,29.13,23.30
10,DET,27.97,26.44
28,SF,27.29,20.32
2,BAL,26.85,23.69
8,DAL,26.22,25.67
5,CHI,25.81,21.42
14,JAX,25.47,18.80
6,CIN,25.19,23.16
15,KC,25.04,19.55
13,IND,24.60,25.90


# Weekly Projected Total

The matchup total uses the same basic preseason scoring concept: each offense is paired with the opposing defense.

Because the weekly offense/defense inputs are already heavily shrunk, no additional Week 1 multiplier is applied here.

The final projected scores are forced to satisfy both:

- projected score difference = weekly expected margin
- projected score sum = weekly projected total


In [14]:
home_scoring_update = scoring_update.rename(
    columns={
        "team": "home_team",
        "current_off_ppg":
            "home_current_off_ppg",
        "current_def_ppg":
            "home_current_def_ppg"
    }
)

away_scoring_update = scoring_update.rename(
    columns={
        "team": "away_team",
        "current_off_ppg":
            "away_current_off_ppg",
        "current_def_ppg":
            "away_current_def_ppg"
    }
)

In [15]:
games = (
    games
    .merge(
        home_scoring_update,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring_update,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

required_scoring_columns = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[
    required_scoring_columns
].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2


assert games["projected_total"].notna().all()

assert np.allclose(
    games["projected_home_score"]
    - games["projected_away_score"],
    games["expected_home_margin"]
)

assert np.allclose(
    games["projected_home_score"]
    + games["projected_away_score"],
    games["projected_total"]
)


display(
    games[
        [
            "away_team",
            "home_team",
            "away_current_off_ppg",
            "home_current_def_ppg",
            "home_current_off_ppg",
            "away_current_def_ppg",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin"
        ]
    ].round(2)
)

,away_team,home_team,away_current_off_ppg,home_current_def_ppg,home_current_off_ppg,away_current_def_ppg,projected_away_score,projected_home_score,projected_total,expected_home_margin
0,PIT,CLE,20.66,23.36,19.23,21.50,22.46,19.91,42.38,-2.55
1,ARI,NYG,22.07,21.61,18.80,25.09,22.36,21.42,43.78,-0.95
2,DAL,HOU,26.22,22.16,20.98,25.67,22.59,24.92,47.51,2.33
3,DEN,SF,22.39,20.32,27.29,21.92,22.18,23.77,45.95,1.59
4,DET,CAR,27.97,25.65,23.88,26.44,28.47,23.50,51.97,-4.98
5,GB,TB,21.62,24.58,22.36,25.75,24.31,22.84,47.15,-1.47
6,IND,WAS,24.60,26.15,23.73,25.90,27.31,22.88,50.19,-4.43
7,JAX,CIN,25.47,23.16,25.19,18.80,23.91,22.41,46.32,-1.50
8,KC,LV,25.04,21.84,23.28,19.55,24.34,20.51,44.85,-3.82
9,LAC,SEA,19.39,19.82,24.42,23.33,17.23,26.25,43.48,9.01


# Market Comparison

nflverse uses positive `spread_line` for a home favorite and negative for an away favorite. The values below are converted to standard sportsbook notation.

Market information is used only after the model projections have been completed.


In [16]:
games["home_spread"] = -games["spread_line"]
games["away_spread"] = games["spread_line"]

games["model_home_spread"] = -games["expected_home_margin"]
games["model_away_spread"] = games["expected_home_margin"]

games["home_ats_edge"] = (
    games["home_spread"]
    - games["model_home_spread"]
)

games["away_ats_edge"] = (
    games["away_spread"]
    - games["model_away_spread"]
)

games["ats_pick"] = np.where(
    games["home_ats_edge"] > games["away_ats_edge"],
    games["home_team"],
    np.where(
        games["away_ats_edge"] > games["home_ats_edge"],
        games["away_team"],
        "PUSH"
    )
)

games["ats_difference"] = (
    games[["home_ats_edge", "away_ats_edge"]]
    .max(axis=1)
)

games["total_difference_signed"] = (
    games["projected_total"]
    - games["total_line"]
)

games["total_difference"] = (
    games["total_difference_signed"].abs()
)

games["total_pick"] = np.where(
    games["total_difference_signed"] > 0,
    "OVER",
    np.where(
        games["total_difference_signed"] < 0,
        "UNDER",
        "PUSH"
    )
)


## Unit Sizing

Unit sizes are assigned from the model's edge over the market.

ATS and totals use separate thresholds because the totals model is newer and requires a larger edge before qualifying as a play.

These units do not change any projections or picks. They provide a consistent way to track performance by model confidence.

In [17]:
def assign_ats_units(edge):
    if pd.isna(edge) or edge < 1.0:
        return 0.0
    elif edge < 2.0:
        return 0.5
    elif edge < 3.0:
        return 1.0
    elif edge < 4.0:
        return 1.5
    else:
        return 2.0


def assign_total_units(edge):
    if pd.isna(edge) or edge < 2.0:
        return 0.0
    elif edge < 3.5:
        return 0.5
    elif edge < 5.0:
        return 1.0
    elif edge < 7.0:
        return 1.5
    else:
        return 2.0


games["ats_units"] = games["ats_difference"].apply(assign_ats_units)

games["total_units"] = games["total_difference"].apply(assign_total_units)


games[
    [
        "away_team",
        "home_team",
        "ats_pick",
        "ats_difference",
        "ats_units",
        "total_pick",
        "total_difference",
        "total_units"
    ]
]

,away_team,home_team,ats_pick,ats_difference,ats_units,total_pick,total_difference,total_units
0,PIT,CLE,PIT,0.051157,0.0,OVER,3.876813,1.0
1,ARI,NYG,NYG,1.552436,0.5,UNDER,0.717585,0.0
2,DAL,HOU,DAL,0.667262,0.0,UNDER,0.988460,0.0
3,DEN,SF,DEN,1.413146,0.5,UNDER,1.546030,0.0
4,DET,CAR,DET,1.477960,0.5,OVER,0.470920,0.0
5,GB,TB,TB,2.029028,1.0,OVER,8.653928,2.0
6,IND,WAS,IND,0.925486,0.0,OVER,2.685660,0.5
7,JAX,CIN,JAX,4.003373,2.0,UNDER,5.181758,1.5
8,KC,LV,LV,0.675199,0.0,UNDER,2.645859,0.5
9,LAC,SEA,SEA,2.011969,1.0,OVER,0.979571,0.0


# Display Helpers


In [18]:
def format_spread(team, line):
    if pd.isna(line):
        return "N/A"
    if line > 0:
        return f"{team} +{line:.1f}"
    return f"{team} {line:.1f}"

def model_spread_display(row):
    if row["model_home_spread"] < 0:
        return format_spread(row["home_team"], row["model_home_spread"])
    if row["model_away_spread"] < 0:
        return format_spread(row["away_team"], row["model_away_spread"])
    return "PICK"

def market_spread_display(row):
    if pd.isna(row["spread_line"]):
        return "N/A"
    if row["home_spread"] < 0:
        return format_spread(row["home_team"], row["home_spread"])
    if row["away_spread"] < 0:
        return format_spread(row["away_team"], row["away_spread"])
    return "PICK"

def ats_pick_display(row):
    if row["ats_pick"] == "PUSH":
        return "PUSH"
    if row["ats_pick"] == row["home_team"]:
        return format_spread(row["home_team"], row["home_spread"])
    return format_spread(row["away_team"], row["away_spread"])

games["projected_score"] = (
    games["away_team"]
    + " "
    + games["projected_away_score"].round(1).astype(str)
    + " - "
    + games["home_team"]
    + " "
    + games["projected_home_score"].round(1).astype(str)
)

games["model_spread_display"] = games.apply(
    model_spread_display, axis=1
)

games["market_spread_display"] = games.apply(
    market_spread_display, axis=1
)

games["ats_pick_display"] = games.apply(
    ats_pick_display, axis=1
)


# Final Week Projection Board


In [19]:
projection_board = pd.DataFrame({
    "Matchup": games["away_team"] + " @ " + games["home_team"],
    "Projected Score": games["projected_score"],
    "SU Pick": games["predicted_winner"],
    "Win Prob": games["predicted_win_probability"],
    "Model Spread": games["model_spread_display"],
    "Market Spread": games["market_spread_display"],
    "ATS Pick": games["ats_pick_display"],
    "ATS Diff": games["ats_difference"],
    "ATS Units": games["ats_units"].map(lambda x: f"{x:.1f}"),
    "Model Total": games["projected_total"],
    "Market Total": games["total_line"],
    "O/U Pick": games["total_pick"],
    "Total Diff": games["total_difference"],
    "Total Units": games["total_units"].map(lambda x: f"{x:.1f}"),
    "Preseason Margin": games["preseason_expected_home_margin"],
    "Weekly Margin": games["expected_home_margin"],
    "Margin Change": games["margin_change_from_preseason"]
})

display(
    projection_board.style.format({
        "Win Prob": "{:.3f}",
        "ATS Diff": "{:.2f}",
        "Model Total": "{:.1f}",
        "Market Total": "{:.1f}",
        "Total Diff": "{:.1f}",
        "Preseason Margin": "{:.2f}",
        "Weekly Margin": "{:.2f}",
        "Margin Change": "{:+.2f}"
    }).hide(axis="index")
)


Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,ATS Units,Model Total,Market Total,O/U Pick,Total Diff,Total Units,Preseason Margin,Weekly Margin,Margin Change
PIT @ CLE,PIT 22.5 - CLE 19.9,PIT,0.571,PIT -2.6,PIT -2.5,PIT -2.5,0.05,0.0,42.4,38.5,OVER,3.9,1.0,-2.93,-2.55,+0.38
ARI @ NYG,ARI 22.4 - NYG 21.4,ARI,0.527,ARI -0.9,ARI -2.5,NYG +2.5,1.55,0.5,43.8,44.5,UNDER,0.7,0.0,1.99,-0.95,-2.93
DAL @ HOU,DAL 22.6 - HOU 24.9,HOU,0.565,HOU -2.3,HOU -3.0,DAL +3.0,0.67,0.0,47.5,48.5,UNDER,1.0,0.0,4.96,2.33,-2.63
DEN @ SF,DEN 22.2 - SF 23.8,SF,0.544,SF -1.6,SF -3.0,DEN +3.0,1.41,0.5,46.0,47.5,UNDER,1.5,0.0,0.56,1.59,+1.03
DET @ CAR,DET 28.5 - CAR 23.5,DET,0.637,DET -5.0,DET -3.5,DET -3.5,1.48,0.5,52.0,51.5,OVER,0.5,0.0,-6.79,-4.98,+1.81
GB @ TB,GB 24.3 - TB 22.8,GB,0.541,GB -1.5,GB -3.5,TB +3.5,2.03,1.0,47.2,38.5,OVER,8.7,2.0,-0.71,-1.47,-0.76
IND @ WAS,IND 27.3 - WAS 22.9,IND,0.622,IND -4.4,IND -3.5,IND -3.5,0.93,0.0,50.2,47.5,OVER,2.7,0.5,-2.53,-4.43,-1.89
JAX @ CIN,JAX 23.9 - CIN 22.4,JAX,0.542,JAX -1.5,CIN -2.5,JAX +2.5,4.00,2.0,46.3,51.5,UNDER,5.2,1.5,-0.69,-1.50,-0.81
KC @ LV,KC 24.3 - LV 20.5,KC,0.606,KC -3.8,KC -4.5,LV +4.5,0.68,0.0,44.9,47.5,UNDER,2.6,0.5,-5.23,-3.82,+1.41
LAC @ SEA,LAC 17.2 - SEA 26.2,SEA,0.737,SEA -9.0,SEA -7.0,SEA -7.0,2.01,1.0,43.5,42.5,OVER,1.0,0.0,5.22,9.01,+3.80


# Biggest Model vs Market Differences


In [20]:
biggest_ats_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values("ATS Diff", ascending=False)
    .head(5)
    .copy()
)

biggest_total_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values("Total Diff", ascending=False)
    .head(5)
    .copy()
)

print("BIGGEST ATS DIFFERENCES")
display(biggest_ats_differences.round(2))

print()
print("BIGGEST TOTAL DIFFERENCES")
display(biggest_total_differences.round(2))


BIGGEST ATS DIFFERENCES


,Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
7,JAX @ CIN,JAX 23.9 - CIN 22.4,JAX -1.5,CIN -2.5,JAX +2.5,4.00
13,NYJ @ CHI,NYJ 19.6 - CHI 25.7,CHI -6.2,CHI -3.5,CHI -3.5,2.66
5,GB @ TB,GB 24.3 - TB 22.8,GB -1.5,GB -3.5,TB +3.5,2.03
9,LAC @ SEA,LAC 17.2 - SEA 26.2,SEA -9.0,SEA -7.0,SEA -7.0,2.01
1,ARI @ NYG,ARI 22.4 - NYG 21.4,ARI -0.9,ARI -2.5,NYG +2.5,1.55



BIGGEST TOTAL DIFFERENCES


,Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
5,GB @ TB,GB 24.3 - TB 22.8,47.15,38.5,OVER,8.65
7,JAX @ CIN,JAX 23.9 - CIN 22.4,46.32,51.5,UNDER,5.18
11,MIA @ MIN,MIA 16.4 - MIN 26.5,42.86,38.5,OVER,4.36
0,PIT @ CLE,PIT 22.5 - CLE 19.9,42.38,38.5,OVER,3.88
14,TEN @ BAL,TEN 16.7 - BAL 28.7,45.34,42.5,OVER,2.84


In [21]:
biggest_ats_differences["ATS Diff"] = (
    biggest_ats_differences["ATS Diff"].round(2)
)

biggest_total_differences["Model Total"] = (
    biggest_total_differences["Model Total"].round(1)
)

biggest_total_differences["Market Total"] = (
    biggest_total_differences["Market Total"].round(1)
)

biggest_total_differences["Total Diff"] = (
    biggest_total_differences["Total Diff"].round(1)
)

# Biggest Changes from the Preseason Projection

This is one of the most important weekly diagnostics. It shows which games changed most because of new in season information.


In [22]:
biggest_projection_changes = (
    projection_board[
        [
            "Matchup",
            "Preseason Margin",
            "Weekly Margin",
            "Margin Change",
            "SU Pick",
            "Win Prob"
        ]
    ]
    .assign(
        Absolute_Change=lambda x: x["Margin Change"].abs()
    )
    .sort_values("Absolute_Change", ascending=False)
    .drop(columns="Absolute_Change")
)

display(biggest_projection_changes.round(3))


,Matchup,Preseason Margin,Weekly Margin,Margin Change,SU Pick,Win Prob
11,MIA @ MIN,4.682,10.086,5.404,MIN,0.761
9,LAC @ SEA,5.216,9.012,3.796,SEA,0.737
1,ARI @ NYG,1.986,-0.948,-2.933,ARI,0.527
2,DAL @ HOU,4.960,2.333,-2.627,HOU,0.565
12,NE @ BUF,4.670,7.264,2.594,BUF,0.696
10,LA @ PHI,-0.728,-3.037,-2.309,LA,0.585
6,IND @ WAS,-2.534,-4.425,-1.891,IND,0.622
4,DET @ CAR,-6.788,-4.978,1.810,DET,0.637
8,KC @ LV,-5.232,-3.825,1.407,KC,0.606
13,NYJ @ CHI,7.464,6.164,-1.301,CHI,0.668


In [23]:
biggest_projection_changes["Preseason Margin"] = (
    biggest_projection_changes["Preseason Margin"].round(2)
)

biggest_projection_changes["Weekly Margin"] = (
    biggest_projection_changes["Weekly Margin"].round(2)
)

biggest_projection_changes["Margin Change"] = (
    biggest_projection_changes["Margin Change"].round(2)
)

biggest_projection_changes["Win Prob"] = (
    biggest_projection_changes["Win Prob"].round(3)
)

# Sanity Checks


In [24]:
assert len(games) == len(target_schedule)
assert games["weekly_neutral_strength_diff"].notna().all()
assert games["expected_home_margin"].notna().all()
assert games["predicted_win_probability"].between(0.5, 1.0).all()
assert games["projected_total"].notna().all()

score_margin_check = (
    games["projected_home_score"]
    - games["projected_away_score"]
)

score_total_check = (
    games["projected_home_score"]
    + games["projected_away_score"]
)

assert np.allclose(
    score_margin_check,
    games["expected_home_margin"]
)

assert np.allclose(
    score_total_check,
    games["projected_total"]
)

print("All weekly game prediction sanity checks passed.")


All weekly game prediction sanity checks passed.


# Save Week Outputs

Everything is written only to the weekly layer.

The detailed parquet preserves all modeling fields for later evaluation. The CSV files provide the clean weekly projection board and largest model versus market differences.


In [25]:
detailed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_projections.parquet"
)

current_games = games.copy()

if detailed_path.exists():

    previous_games = pd.read_parquet(
        detailed_path
    )

    current_game_ids = set(
        current_games["game_id"]
    )

    frozen_games = previous_games[
        ~previous_games["game_id"].isin(
            current_game_ids
        )
    ].copy()

    detailed_save = pd.concat(
        [
            frozen_games,
            current_games
        ],
        ignore_index=True
    )

else:

    detailed_save = current_games.copy()


detailed_save.to_parquet(
    detailed_path,
    index=False
)

projection_save = projection_board.copy()

projection_save["ATS Units"] = games["ats_units"].values
projection_save["Total Units"] = games["total_units"].values

projection_save["Win Prob"] = (
    projection_save["Win Prob"].round(3)
)

projection_save["ATS Diff"] = (
    projection_save["ATS Diff"].round(2)
)

projection_save["ATS Units"] = (
    projection_save["ATS Units"].round(1)
)

projection_save["Model Total"] = (
    projection_save["Model Total"].round(1)
)

projection_save["Market Total"] = (
    projection_save["Market Total"].round(1)
)

projection_save["Total Diff"] = (
    projection_save["Total Diff"].round(1)
)

projection_save["Total Units"] = (
    projection_save["Total Units"].round(1)
)

projection_save["Preseason Margin"] = (
    projection_save["Preseason Margin"].round(2)
)

projection_save["Weekly Margin"] = (
    projection_save["Weekly Margin"].round(2)
)

projection_save["Margin Change"] = (
    projection_save["Margin Change"].round(2)
)

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

if projection_csv_path.exists():

    previous_projections = pd.read_csv(
        projection_csv_path
    )

    current_matchups = set(
        projection_save["Matchup"]
    )

    frozen_projections = previous_projections[
        ~previous_projections["Matchup"].isin(
            current_matchups
        )
    ].copy()

    projection_save = pd.concat(
        [
            frozen_projections,
            projection_save
        ],
        ignore_index=True
    )

projection_save.to_csv(
    projection_csv_path,
    index=False
)

biggest_ats_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv",
    index=False
)

biggest_total_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv",
    index=False
)

biggest_projection_changes.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv",
    index=False
)


print("Saved:")
print(detailed_path)
print(projection_csv_path)

print(
    f"\nTotal saved Week {TARGET_WEEK} projections: "
    f"{len(projection_save)}"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv"
)

Saved:
..\..\data\processed\weekly\week_04_projections.parquet
..\..\weekly_projections\outputs\week_04\week_04_projections.csv

Total saved Week 4 projections: 16
..\..\weekly_projections\outputs\week_04\week_04_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_04\week_04_biggest_total_differences.csv
..\..\weekly_projections\outputs\week_04\week_04_biggest_projection_changes.csv


# Weekly Pipeline

For each future week, the intended order is:

`02_Weekly_Data_Update`  
→ `03_Inseason_Feature_Engineering`  
→ `04_Weekly_Team_Strength`  
→ `06_Week2_Projections`

The original preseason notebooks remain frozen throughout the season.

The next notebook will be `06_Weekly_Evaluation.ipynb`, which will compare locked weekly predictions against actual results after each week is complete.
